In [ ]:
import sys, os, glob, shutil, time, json, gc
import numpy as np, pandas as pd, torch
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:7.1f}с] {m}", flush=True)
log(f"GPU: {torch.cuda.get_device_name(0)}, ядер CPU: {os.cpu_count()}")

os.makedirs("/kaggle/working/src", exist_ok=True); os.makedirs("/kaggle/working/models", exist_ok=True)
code = os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(code + "/*.json"): shutil.copy(p, "/kaggle/working/models/")
struct = os.path.dirname(glob.glob("/kaggle/input/**/pair_boost_hybrid.npz", recursive=True)[0])
for p in glob.glob(struct + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(struct + "/*.npz"): shutil.copy(p, "/kaggle/working/models/")
fus = os.path.dirname(glob.glob("/kaggle/input/**/fusion_boost.npz", recursive=True)[0])
for p in glob.glob(fus + "/fusion_*"): shutil.copy(p, "/kaggle/working/models/")
open("/kaggle/working/src/__init__.py", "a").close()

# Энкодеры: fp16 там, где есть, иначе fp32 — на скорость это не влияет, модель всё равно
# грузится в fp16, важен только состав.
for prefix, tag in (("ce_relaxed16","ce_relaxed"), ("ce_e516","ce_e5"), ("ce_ru16","ce_ru"),
                    ("ce_combo","ce_combo"), ("ce_spec","ce_spec"), ("ce_self","ce_self")):
    hits = glob.glob(f"/kaggle/input/**/{prefix}__model.safetensors", recursive=True)
    if not hits: log(f"НЕТ ВЕСОВ: {prefix}"); continue
    root = os.path.dirname(hits[0]); dst = f"/kaggle/working/models/{tag}"
    os.makedirs(dst, exist_ok=True)
    for f in ("model.safetensors","config.json","tokenizer.json","tokenizer_config.json","inference_config.json"):
        if not os.path.exists(f"{dst}/{f}"): os.symlink(f"{root}/{prefix}__{f}", f"{dst}/{f}")
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")

big = os.path.dirname(glob.glob("/kaggle/input/**/llm_pairs_2m.parquet", recursive=True)[0])
pairs = pd.read_parquet(big + "/llm_pairs_2m.parquet").sample(120_000, random_state=1).reset_index(drop=True)
items = pd.read_parquet(big + "/llm_items_2m.parquet")
used = pd.unique(np.concatenate([pairs.id1.to_numpy(), pairs.id2.to_numpy()]))
items = items[items.id.isin(used)].reset_index(drop=True)
matches = pairs[["id1","id2"]]
log(f"замер на пар {len(matches):,}, карточек {len(items):,} — всё линейно, пересчёт ниже")

# Вызываем ровно то, что вызывает run.py на проверке: тогда замер учитывает и загрузку,
# и сборку текстов, и запись результата — всё, за что идёт время на самом деле.
matches.to_parquet("/kaggle/working/test_matches.parquet", index=False)
items.to_parquet("/kaggle/working/test_items.parquet", index=False)
N = len(matches)
del pairs, items, matches; gc.collect()

from src.pipeline import predict_pipeline
t = time.perf_counter()
predict_pipeline(items_path="/kaggle/working/test_items.parquet",
                 matches_path="/kaggle/working/test_matches.parquet",
                 output_path="/kaggle/working/submit.csv",
                 method="blend")
total = time.perf_counter() - t
out = pd.read_csv("/kaggle/working/submit.csv")
log(f"\nВЕСЬ ПРОГОН: {total:.1f}с на {N:,} пар, строк на выходе {len(out):,}")
log(f"  Private (255 тыс.): {total*255000/N:.0f}с при лимите 780 — "
    + ("ВЛЕЗАЕМ" if total*255000/N < 780 else "НЕ ВЛЕЗАЕМ"))
log(f"  Public  (110 тыс.): {total*110000/N:.0f}с при лимите 360 — "
    + ("ВЛЕЗАЕМ" if total*110000/N < 360 else "НЕ ВЛЕЗАЕМ"))
log("  замер на T4; на H100 быстрее станут только стадии видеокарты")
log(f"  скор: мин {out.predict.min():.4f} макс {out.predict.max():.4f}, пустых {int(out.predict.isna().sum())}")
